# GDELT Extraction — arbitrary year range (clean rebuild)

Run once per range you need. **Only the config cell below should need editing between runs** --
everything downstream (paths, scoring, output filenames) derives from `START_YEAR`/`END_YEAR`
automatically, so a 2024-2025 run needs zero other edits.

**Fixes folded in from the 2015-2016 run's actual failures, so they don't recur:**
- Every path is `../../data/...`, matching running this from `notebooks/01_data_extraction/`.
  The earlier bare `"data/..."` version silently wrote into a phantom nested
  `notebooks/01_data_extraction/data/` folder instead of the real one.
- The final combine step (in `pull_gdelt_years()`) now streams month-by-month straight to a
  `ParquetWriter` instead of `pd.concat()`-ing every month into memory at once -- the
  concat version is what caused the earlier RAM spike.
- Merge-into-a-combined-file is now **optional and off by default** (`DO_MERGE = False`).
  Since retraining isn't happening, standalone per-range files are all that's needed --
  see chat for the reasoning. Flip it on only if a future phase needs a merged training pool.
- `FULL_RANGE_LO`/`FULL_RANGE_HI` for the scoring step now default to `START_YEAR`/`END_YEAR`
  automatically -- no separate manual sync needed between the pull config and the scoring config.

**Hard constraint:** GDELT's daily-file feed starts 2015-02-19. `START_YEAR < 2015` raises
immediately. `START_YEAR == 2015` runs, but Jan 1 - Feb 18 has zero events -- a real gap in
GDELT's own coverage, flagged explicitly in the final check cell.

In [3]:
import os, io, zipfile, time, sys
import pandas as pd
import numpy as np
import requests
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timedelta
from tqdm import tqdm

# ============================================================
# ---- CONFIG -- the only cell that should need editing per run ----
START_YEAR, END_YEAR = 2024, 2025
# ============================================================

COUNTRIES = None
EVENT_ROOT_CODES = None
MIN_MENTIONS = 5
MIN_SOURCES = 2
DO_MERGE = False   # flip to True + fill in BASE_EVENTS_FILE below only if a merged training pool is needed later

RAW_DIR = os.path.join("..", "..", "data", "raw")
INTERIM_DIR = os.path.join("..", "..", "data", "interim")
CACHE_DIR = os.path.join("..", "..", "data", "cache")
UTILS_DIR = os.path.join("..", "..", "src", "utils")
sys.path.append(UTILS_DIR)

CHECKPOINT_DIR = os.path.join(CACHE_DIR, f"gdelt_chunks_{START_YEAR}_{END_YEAR}")
OUTPUT_PATH_SLICE = os.path.join(INTERIM_DIR, f"gdelt_events_{START_YEAR}_{END_YEAR}.parquet")
BASE_EVENTS_FILE = os.path.join(INTERIM_DIR, "gdelt_events_2017_2024.parquet")   # only used if DO_MERGE=True
MERGED_EVENTS_OUT = os.path.join(INTERIM_DIR, f"gdelt_events_merged_{START_YEAR}_{END_YEAR}.parquet")

FULL_RANGE_LO, FULL_RANGE_HI = START_YEAR, END_YEAR   # scoring step defaults to this run's own range

GDELT_DAILY_FEED_START = datetime(2015, 2, 19)

if START_YEAR < 2015:
    raise ValueError(
        f"START_YEAR={START_YEAR} is before GDELT's daily-file feed begins (2015-02-19). "
        f"This pipeline cannot pull pre-2015 data."
    )
if START_YEAR == 2015:
    print("NOTE: 2015 is a partial year in GDELT -- Jan 1 - Feb 18, 2015 will have zero events. "
          "This is a real gap in GDELT's own coverage, not an extraction bug.")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(INTERIM_DIR, exist_ok=True)
print(f"range: {START_YEAR}-{END_YEAR}")
print(f"CHECKPOINT_DIR -> {CHECKPOINT_DIR}")
print(f"OUTPUT_PATH_SLICE -> {OUTPUT_PATH_SLICE}")

range: 2024-2025
CHECKPOINT_DIR -> ../../data/cache/gdelt_chunks_2024_2025
OUTPUT_PATH_SLICE -> ../../data/interim/gdelt_events_2024_2025.parquet


In [4]:
GDELT_COLUMNS_V1 = [
    "GLOBALEVENTID","SQLDATE","MonthYear","Year","FractionDate",
    "Actor1Code","Actor1Name","Actor1CountryCode","Actor1KnownGroupCode",
    "Actor1EthnicCode","Actor1Religion1Code","Actor1Religion2Code",
    "Actor1Type1Code","Actor1Type2Code","Actor1Type3Code",
    "Actor2Code","Actor2Name","Actor2CountryCode","Actor2KnownGroupCode",
    "Actor2EthnicCode","Actor2Religion1Code","Actor2Religion2Code",
    "Actor2Type1Code","Actor2Type2Code","Actor2Type3Code",
    "IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass",
    "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone",
    "Actor1Geo_Type","Actor1Geo_FullName","Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code","Actor1Geo_Lat","Actor1Geo_Long","Actor1Geo_FeatureID",
    "Actor2Geo_Type","Actor2Geo_FullName","Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code","Actor2Geo_Lat","Actor2Geo_Long","Actor2Geo_FeatureID",
    "ActionGeo_Type","ActionGeo_FullName","ActionGeo_CountryCode",
    "ActionGeo_ADM1Code","ActionGeo_Lat","ActionGeo_Long","ActionGeo_FeatureID",
    "DATEADDED","SOURCEURL",
]

KEEP_COLS = [
    "GLOBALEVENTID","SQLDATE","DATEADDED","Actor1Name","Actor1CountryCode",
    "Actor2Name","Actor2CountryCode","ActionGeo_CountryCode","ActionGeo_FullName",
    "EventCode","EventBaseCode","EventRootCode",
    "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone","SOURCEURL",
]

## Date helpers, downloader, filter

In [5]:
def generate_gdelt_dates(start_date, end_date):
    cur = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    cur = max(cur, GDELT_DAILY_FEED_START)
    days = []
    while cur <= end:
        days.append(cur.strftime("%Y%m%d")); cur += timedelta(days=1)
    return days

def month_ranges(y0, y1):
    out, cur, end = [], datetime(y0,1,1), datetime(y1,12,31)
    while cur <= end:
        nxt = datetime(cur.year + (cur.month==12), (cur.month % 12)+1, 1)
        out.append((cur.strftime("%Y-%m-%d"), (nxt-timedelta(days=1)).strftime("%Y-%m-%d")))
        cur = nxt
    return out

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 (research data pull)"})

def download_gdelt_file_v1(date_str, retries=4, pause=0.4):
    url = f"http://data.gdeltproject.org/events/{date_str}.export.CSV.zip"
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=60)
            if r.status_code == 200:
                z = zipfile.ZipFile(io.BytesIO(r.content))
                raw = pd.read_csv(z.open(z.namelist()[0]), sep="\t",
                                  header=None, dtype=str, low_memory=False)
                if raw.shape[1] != len(GDELT_COLUMNS_V1):
                    print(f"  \u26a0 {date_str}: {raw.shape[1]} cols (expected {len(GDELT_COLUMNS_V1)})")
                    return None
                raw.columns = GDELT_COLUMNS_V1
                time.sleep(pause)
                return raw
            if r.status_code == 404:
                return None
            print(f"  {date_str}: HTTP {r.status_code} (throttled?), retry {attempt+1}/{retries}")
            time.sleep(2 ** attempt + 1)
        except Exception as e:
            print(f"  {date_str}: {type(e).__name__}, retry {attempt+1}/{retries}")
            time.sleep(2 ** attempt + 1)
    print(f"  {date_str}: gave up after {retries} tries")
    return None

def filter_events(df, countries=COUNTRIES, event_root_codes=EVENT_ROOT_CODES,
                   min_mentions=MIN_MENTIONS, min_sources=MIN_SOURCES):
    df = df.copy()
    for c in ["NumMentions","NumSources","NumArticles","GoldsteinScale","AvgTone"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    if countries is not None:
        df = df[
            df["Actor1CountryCode"].isin(countries) |
            df["Actor2CountryCode"].isin(countries) |
            df["ActionGeo_CountryCode"].isin(countries)
        ]
    if df.empty: return pd.DataFrame()
    if event_root_codes is not None:
        df = df[df["EventRootCode"].isin(event_root_codes)]
    if df.empty: return pd.DataFrame()
    df = df[(df["NumMentions"] >= min_mentions) & (df["NumSources"] >= min_sources)]
    if df.empty: return pd.DataFrame()
    return df[[c for c in KEEP_COLS if c in df.columns]]

## Pull functions -- final combine step now streams to disk instead of concatenating in memory

In [6]:
def extract_events_for_range(start_date, end_date, output_path):
    all_events = []
    for d in tqdm(generate_gdelt_dates(start_date, end_date)):
        df = download_gdelt_file_v1(d)
        if df is None or df.empty:
            continue
        filt = filter_events(df)
        if not filt.empty:
            all_events.append(filt)
    if not all_events:
        print("No matching events found."); return pd.DataFrame()
    result = pd.concat(all_events, ignore_index=True)   # one month's worth -- small, fine to concat
    result["date"] = pd.to_datetime(result["SQLDATE"], format="%Y%m%d", errors="coerce")
    result = result.drop_duplicates(subset=["GLOBALEVENTID"])
    result.to_parquet(output_path, index=False)
    print(f"Saved {len(result):,} events to {output_path}")
    return result


def pull_gdelt_years(y0=START_YEAR, y1=END_YEAR, out_path=None):
    '''Downloads month-by-month (resumable -- skips months whose chunk file already exists),
    then streams every chunk straight to a single ParquetWriter -- never holds more than one
    month's dataframe in memory at once, unlike a pd.concat()-based combine.
    '''
    out_path = out_path or OUTPUT_PATH_SLICE
    chunk_files = []
    for start, end in month_ranges(y0, y1):
        tag = start[:7]
        out = os.path.join(CHECKPOINT_DIR, f"gdelt_{tag}.parquet")
        chunk_files.append(out)
        if os.path.exists(out):
            print(f"\u2713 {tag} done \u2014 skipping"); continue
        print(f"\n=== {tag} ===")
        extract_events_for_range(start, end, output_path=out)

    writer, total = None, 0
    for f in chunk_files:
        if not os.path.exists(f):
            continue
        df = pd.read_parquet(f)
        total += len(df)
        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)
        del df, table
    if writer:
        writer.close()

    if total == 0:
        print("No chunk files produced -- nothing to combine.")
        return pd.DataFrame()
    print(f"\nFINAL: {total:,} events -> {out_path}")
    return pd.read_parquet(out_path)   # returned for the sanity-check cell below; chunk files already released

## Test on one week before committing to the full range

In [7]:
_test_start = max(datetime(START_YEAR,1,1), GDELT_DAILY_FEED_START).strftime("%Y-%m-%d")
_test_end = (max(datetime(START_YEAR,1,1), GDELT_DAILY_FEED_START) + timedelta(days=6)).strftime("%Y-%m-%d")
test = extract_events_for_range(_test_start, _test_end, output_path=os.path.join(CACHE_DIR, "test_events_range.parquet"))
test.shape

100%|██████████| 7/7 [00:17<00:00,  2.45s/it]


Saved 134,785 events to ../../data/cache/test_events_range.parquet


(134785, 19)

## Full pull for this range

In [8]:
events_slice = pull_gdelt_years(START_YEAR, END_YEAR)
len(events_slice)


=== 2024-01 ===


100%|██████████| 31/31 [01:23<00:00,  2.71s/it]


Saved 707,739 events to ../../data/cache/gdelt_chunks_2024_2025/gdelt_2024-01.parquet

=== 2024-02 ===


100%|██████████| 29/29 [01:28<00:00,  3.06s/it]


Saved 716,806 events to ../../data/cache/gdelt_chunks_2024_2025/gdelt_2024-02.parquet

=== 2024-03 ===


...

100%|██████████| 30/30 [01:16<00:00,  2.53s/it]


Saved 533,587 events to ../../data/cache/gdelt_chunks_2024_2025/gdelt_2025-11.parquet

=== 2025-12 ===


100%|██████████| 31/31 [01:13<00:00,  2.38s/it]


Saved 503,029 events to ../../data/cache/gdelt_chunks_2024_2025/gdelt_2025-12.parquet

FINAL: 15,215,063 events -> ../../data/interim/gdelt_events_2024_2025.parquet


15215063

## Optional: merge into a running combined file

**Skip unless a merged training pool is actually needed** (`DO_MERGE = False` by default,
set in the config cell). `OUTPUT_PATH_SLICE` above is already the finished standalone output
for eval-only years.

In [9]:
if DO_MERGE and len(events_slice):
    existing_events = pd.read_parquet(BASE_EVENTS_FILE)
    print("existing:", len(existing_events), "| years:",
          sorted(pd.to_datetime(existing_events['date']).dt.year.unique()))

    combined_events = pd.concat([existing_events, events_slice], ignore_index=True)
    combined_events = combined_events.drop_duplicates(subset=["GLOBALEVENTID"])
    combined_events.to_parquet(MERGED_EVENTS_OUT, index=False)
    print(f"\nCombined: {len(combined_events):,} events -> {MERGED_EVENTS_OUT}")
    print("years:", sorted(pd.to_datetime(combined_events['date']).dt.year.unique()))
else:
    print("Merge skipped (DO_MERGE is False, or no events were pulled for this range).")

Merge skipped (DO_MERGE is False, or no events were pulled for this range).


## Scoring/aggregation -- turns raw events into the country-year/pair-year feature files

Uses `FULL_RANGE_LO`/`FULL_RANGE_HI` (already set to `START_YEAR`/`END_YEAR` in the config cell
above) and reads from `OUTPUT_PATH_SLICE` -- the standalone file from this run, not a merged one.

In [10]:
from country_codes import ISO3_2_M49

SCORE_COLS = ["GLOBALEVENTID","Actor1CountryCode","Actor2CountryCode",
              "EventRootCode","GoldsteinScale","NumMentions","AvgTone","date"]
CRIT = ["14","17","18","19","20"]
K = 10
iso2m49 = {**ISO3_2_M49, "TWN": "490"}

def stream_score(path=None):
    path = path or OUTPUT_PATH_SLICE
    pf = pq.ParquetFile(path)
    node_acc, pair_acc = {}, {}
    node_topk, pair_topk = {}, {}

    def add(acc, key, score, gold, ment, tone):
        s = acc.get(key)
        if s is None:
            acc[key] = [1, score, score, gold*max(ment,1), max(ment,1), tone]
        else:
            s[0]+=1; s[1]+=score; s[2]=max(s[2],score); s[3]+=gold*max(ment,1); s[4]+=max(ment,1); s[5]+=tone

    def add_topk(acc, key, score, gold, ment, tone):
        buf = acc.setdefault(key, [])
        buf.append((score, gold, ment, tone))
        if len(buf) > 200:
            buf.sort(key=lambda x:-x[0]); del buf[100:]

    for b in range(pf.num_row_groups):
        ch = pf.read_row_group(b, columns=SCORE_COLS).to_pandas()
        ch = ch.dropna(subset=["Actor1CountryCode","date"])
        ch["EventRootCode"] = ch["EventRootCode"].astype(str)
        ch = ch[ch["EventRootCode"].isin(CRIT)]
        if ch.empty: continue
        ch["ym"] = pd.to_datetime(ch["date"], errors="coerce").dt.to_period("M")
        ch = ch.dropna(subset=["ym"])
        g = pd.to_numeric(ch["GoldsteinScale"], errors="coerce").fillna(0).values
        m = pd.to_numeric(ch["NumMentions"], errors="coerce").fillna(0).values
        t = pd.to_numeric(ch["AvgTone"], errors="coerce").fillna(0).values
        sc = np.abs(g) * m
        a1 = ch["Actor1CountryCode"].values; a2 = ch["Actor2CountryCode"].values
        ym = ch["ym"].values
        for i in range(len(ch)):
            add(node_acc, (a1[i], ym[i]), sc[i], g[i], m[i], t[i])
            add_topk(node_topk, (a1[i], ym[i]), sc[i], g[i], m[i], t[i])
            if pd.notna(a2[i]):
                add(pair_acc, (a1[i], a2[i], ym[i]), sc[i], g[i], m[i], t[i])
                add_topk(pair_topk, (a1[i], a2[i], ym[i]), sc[i], g[i], m[i], t[i])
        if b % 10 == 0:
            print(f"  row-group {b}/{pf.num_row_groups}")
    return node_acc, pair_acc, node_topk, pair_topk

node_acc, pair_acc, node_topk, pair_topk = stream_score()
print("done streaming.")

  row-group 0/24
  row-group 10/24
  row-group 20/24
done streaming.


In [11]:
def _finalize_node(acc, topk, out, use_topk, year_lo=FULL_RANGE_LO, year_hi=FULL_RANGE_HI):
    rows=[]
    for key,val in (topk.items() if use_topk else acc.items()):
        iso, ym = key
        if use_topk:
            buf = sorted(val, key=lambda x:-x[0])[:K]
            if not buf: continue
            n=len(buf); ssum=sum(x[0] for x in buf); smax=max(x[0] for x in buf)
            gm=sum(x[1]*max(x[2],1) for x in buf); msum=sum(max(x[2],1) for x in buf); tsum=sum(x[3] for x in buf)
        else:
            n,ssum,smax,gm,msum,tsum = val
        rows.append((iso, ym.year, n, ssum/n, smax, gm/max(msum,1), tsum/n))
    df = pd.DataFrame(rows, columns=["iso3","year","n_events","score_mean","score_max","goldstein_wmean","tone_mean"])
    ann = df.groupby(["iso3","year"]).agg(
        events_total=("n_events","sum"), score_mean=("score_mean","mean"),
        score_max=("score_max","max"), score_vol=("score_mean","std"),
        goldstein_wmean=("goldstein_wmean","mean"), tone_mean=("tone_mean","mean"),
        active_months=("n_events", lambda s:(s>0).sum())).reset_index()
    ann = ann[ann["year"].between(year_lo, year_hi)]
    ann["reporterCode"]=ann["iso3"].map(iso2m49); ann=ann.dropna(subset=["reporterCode"])
    ann["reporterCode"]=ann["reporterCode"].astype(int)
    ann.to_parquet(out,index=False); print("saved",out,ann.shape)

_finalize_node(node_acc, node_topk, os.path.join(INTERIM_DIR, f"gdelt_features_by_country_year_{FULL_RANGE_LO}_{FULL_RANGE_HI}.parquet"), use_topk=False)
_finalize_node(node_acc, node_topk, os.path.join(INTERIM_DIR, f"gdelt_features_topk_{FULL_RANGE_LO}_{FULL_RANGE_HI}.parquet"),            use_topk=True)


def _finalize_pair(acc, topk, out, use_topk, year_lo=FULL_RANGE_LO, year_hi=FULL_RANGE_HI):
    rows=[]
    items = topk.items() if use_topk else acc.items()
    for key, val in items:
        a1, a2, ym = key
        if ym.year < year_lo or ym.year > year_hi:
            continue
        if use_topk:
            buf = sorted(val, key=lambda x:-x[0])[:K]
            if not buf: continue
            n=len(buf); ssum=sum(x[0] for x in buf); smax=max(x[0] for x in buf)
            gm=sum(x[1]*max(x[2],1) for x in buf); msum=sum(max(x[2],1) for x in buf)
        else:
            n,ssum,smax,gm,msum,tsum = val
        rows.append((a1, a2, ym.year, n, ssum/n, smax, gm/max(msum,1)))
    df = pd.DataFrame(rows, columns=["Actor1CountryCode","Actor2CountryCode","year",
                                     "pair_events","pair_score_mean","pair_score_max","pair_gold_mean"])
    ann = df.groupby(["Actor1CountryCode","Actor2CountryCode","year"]).agg(
        pair_events=("pair_events","sum"),
        pair_score_mean=("pair_score_mean","mean"),
        pair_score_max=("pair_score_max","max"),
        pair_gold_mean=("pair_gold_mean","mean")).reset_index()
    ann["reporterCode"]=ann["Actor1CountryCode"].map(iso2m49)
    ann["partnerCode"] =ann["Actor2CountryCode"].map(iso2m49)
    ann=ann.dropna(subset=["reporterCode","partnerCode"])
    ann["reporterCode"]=ann["reporterCode"].astype(int)
    ann["partnerCode"] =ann["partnerCode"].astype(int)
    ann.to_parquet(out, index=False); print("saved", out, ann.shape)

_finalize_pair(pair_acc, pair_topk, os.path.join(INTERIM_DIR, f"gdelt_bilateral_by_pair_year_{FULL_RANGE_LO}_{FULL_RANGE_HI}.parquet"), use_topk=False)
_finalize_pair(pair_acc, pair_topk, os.path.join(INTERIM_DIR, f"gdelt_bilateral_topk_{FULL_RANGE_LO}_{FULL_RANGE_HI}.parquet"),         use_topk=True)

saved ../../data/interim/gdelt_features_by_country_year_2024_2025.parquet (393, 10)
saved ../../data/interim/gdelt_features_topk_2024_2025.parquet (393, 10)
saved ../../data/interim/gdelt_bilateral_by_pair_year_2024_2025.parquet (9669, 9)
saved ../../data/interim/gdelt_bilateral_topk_2024_2025.parquet (9669, 9)


## Final check

In [12]:
for f in [os.path.join(INTERIM_DIR, f"gdelt_features_by_country_year_{FULL_RANGE_LO}_{FULL_RANGE_HI}.parquet"),
          os.path.join(INTERIM_DIR, f"gdelt_features_topk_{FULL_RANGE_LO}_{FULL_RANGE_HI}.parquet"),
          os.path.join(INTERIM_DIR, f"gdelt_bilateral_by_pair_year_{FULL_RANGE_LO}_{FULL_RANGE_HI}.parquet"),
          os.path.join(INTERIM_DIR, f"gdelt_bilateral_topk_{FULL_RANGE_LO}_{FULL_RANGE_HI}.parquet")]:
    d = pd.read_parquet(f)
    print(f"{f}: {d.shape}  years={sorted(d['year'].unique())}")

if FULL_RANGE_LO <= 2015:
    print("\nReminder: 2015's events_total will be lower than a full year's worth by construction "
          "(Jan 1 - Feb 18, 2015 has no GDELT coverage) -- don't read a low 2015 score as a real "
          "low-tension year without checking this first.")

../../data/interim/gdelt_features_by_country_year_2024_2025.parquet: (393, 10)  years=[2024, 2025]
../../data/interim/gdelt_features_topk_2024_2025.parquet: (393, 10)  years=[2024, 2025]
../../data/interim/gdelt_bilateral_by_pair_year_2024_2025.parquet: (9669, 9)  years=[2024, 2025]
../../data/interim/gdelt_bilateral_topk_2024_2025.parquet: (9669, 9)  years=[2024, 2025]
